# Attemp

In [ ]:
!pip install transformers accelerate evaluate datasets peft -q

In [ ]:
from transformers import AutoModelForImageClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms, datasets, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split, StratifiedKFold
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold

In [ ]:
import glob
import cv2
import numpy as np
import torch
import nibabel as nib
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.transforms.functional import normalize
import torch.nn.functional as F
from torch.utils.data import random_split
import matplotlib.pyplot as plt
import random
from PIL import Image
import os
from torchvision.transforms import v2
from torchvision.io import read_image
import copy

In [ ]:
model_checkpoint = "google/vit-base-patch16-224"

In [ ]:
def read_data(data_path):
    
    file_list = glob.glob(data_path + '*') # this searches for ALL directories
    print(file_list)
    data = []
    for class_path in file_list:
        class_name = class_path.split("/")[-1]
        for vol_path in glob.glob(class_path + "/*.png"):
            data.append([vol_path, class_name])
    print(len(data))
    return data

In [ ]:
# build dataloaders with pytorch
class MRIDataset(Dataset):
    
    def __init__(self,data,transform=None) -> None:
        
        self.transform = transform
        self.data = [(el, 'main') for el in data]
        
        self.normalization = v2.Compose([
                v2.ToImage(),
                v2.Resize((224,224)),
                v2.ToDtype(torch.float32, scale=True),
                v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # typically from ImageNet
            ])
        
        if transform:
            self.data.extend([(el, 'augm1') for el in data if el[1] == 'AD']) #just for AD samples

            self.transform1 = v2.Compose([
                    v2.RandomRotation(degrees = 3)   #[-3,3]
                ])

        self.N_AD = 0
        self.N_MCI = 0
        self.N_CN = 0
        
        self.class_map = {'AD':0 , 'MCI':1 , 'CN' : 2}
        
        super().__init__()

    def __len__(self):
#         print("AD, MCI and CN numbers are:" , self.N_AD, self.N_MCI , self.N_CN)
        return len(self.data)

    def __getitem__(self, idx):
        
        vol_path, class_name = self.data[idx][0] # data now is a tuple first element is data path
        volume = read_image(vol_path) # C×H×W  [0-255]
        
        class_index = self.class_map[class_name]

        if class_index == 0:
            self.N_AD += 1
            
        if class_index == 1:
            self.N_MCI += 1
            
        if class_index == 2:
            self.N_CN += 1
            
        if self.transform and (self.data[idx][1] == 'augm1'): # I want to double samples in AD group to make the dataset balanced.
            augmented = self.transform1(volume)
            volume = augmented
            
        volume = self.normalization(volume) # for all of training and validation
        
        return volume , class_index

In [ ]:
data_path = '/kaggle/input/middleslices3way/MiddleSlices/'
data_items = read_data(data_path)

In [ ]:
label2id = {label: idx for idx, label in enumerate(sorted(set([item[1] for item in data_items])))}

In [ ]:
label2id = {'AD':'0' , 'MCI':'1' , 'CN':'2'}
id2label = {'0':'AD','1':'MCI','2':'CN'}

In [ ]:
id2label

In [ ]:
model = AutoModelForImageClassification.from_pretrained(
    model_checkpoint,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,  # provide this in case you're planning to fine-tune an already fine-tuned checkpoint
)

In [ ]:
model.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(768, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 3),
            nn.Softmax(dim=1)
        )

In [ ]:
model

In [ ]:
def print_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param:.2f}"
    )

In [ ]:
print_trainable_parameters(model)

In [ ]:
# Print the model architecture to inspect the names
for name, _ in model.named_modules():
    print(name)

In [ ]:
# Define the config for LoRA adaptation
config = LoraConfig(
    r=4,
    lora_alpha=4,
    target_modules = ['key','query','value'],
#     target_modules=["vit.encoder.layer.10.attention.attention.key", "vit.encoder.layer.10.attention.attention.query", "vit.encoder.layer.10.attention.attention.value","vit.encoder.layer.11.attention.attention.key", "vit.encoder.layer.11.attention.attention.query", "vit.encoder.layer.11.attention.attention.value"],
    lora_dropout=0.01,
    bias="none",
    modules_to_save=["classifier"],
#     layers_to_transform=[10, 11],  # Apply LoRA to layers 10 and 11
#     layers_pattern="encoder.layer.{layer}",  # Pattern to locate encoder layers
)

lora_model = get_peft_model(model, config)

print_trainable_parameters(lora_model)

In [ ]:
lora_model

In [ ]:
# Count the number of trainable parameters
trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import KFold

In [ ]:
# Configuration options
k_folds = 5
num_epochs = 100
criterion = nn.CrossEntropyLoss()
learning_rate = 1e-3
batch_size = 32

# For fold results
results = []
metrics = []
best_model_metrics = None
best_model_state = None
best_fold = None
best_val_accuracy = 0

# Set fixed random number seed
torch.manual_seed(42)

dataset = data_items
# Define the K-fold Cross Validator
kfold = KFold(n_splits=k_folds, shuffle=True)

# Start print
print('--------------------------------')

# K-fold Cross Validation model evaluation
for fold, (train_ids, test_ids) in enumerate(kfold.split(dataset)):
    print(f'FOLD {fold}')
    print('--------------------------------')
    
    train_data = [data_items[i] for i in train_ids]
    validation_data = [data_items[i] for i in test_ids]
    
    train_dataset = MRIDataset(train_data, transform=True)
    valid_dataset = MRIDataset(validation_data, transform=False)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

    model = AutoModelForImageClassification.from_pretrained(
        model_checkpoint,
        label2id=label2id,
        id2label=id2label,
        ignore_mismatched_sizes=True,  # provide this in case you're planning to fine-tune an already fine-tuned checkpoint
    )
    model.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(768, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, 3),
        nn.Softmax(dim=1)
    )
    ## we set config parameters upper ##
    lora_model = get_peft_model(model, config)
    model = lora_model.cuda()
    
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate)
    
    train_loss, val_loss = [], []
    train_accuracy, val_accuracy = [], []
    
    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.cuda(), labels.cuda()
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs.logits, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        epoch_loss = running_loss / len(train_dataset)
        epoch_acc = correct / total
        train_loss.append(epoch_loss)
        train_accuracy.append(epoch_acc)

        model.eval()
        running_loss, correct, total = 0.0, 0, 0
        y_true, y_pred, y_scores = [], [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.cuda(), labels.cuda()
                outputs = model(inputs)
                loss = criterion(outputs.logits, labels)
                
                running_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.logits, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(predicted.cpu().numpy())
                y_scores.extend(outputs.logits.softmax(dim=1).cpu().numpy())
        
        epoch_loss = running_loss / len(valid_dataset)
        epoch_acc = correct / total
        val_loss.append(epoch_loss)
        val_accuracy.append(epoch_acc)

        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss[-1]:.4f}, Train Acc: {train_accuracy[-1]:.4f}, Val Loss: {val_loss[-1]:.4f}, Val Acc: {val_accuracy[-1]:.4f}")
    
    # Calculate metrics for the fold
    fold_metrics = {
        "accuracy": epoch_acc,
        "precision": precision_score(y_true, y_pred, average="weighted"),
        "recall": recall_score(y_true, y_pred, average="weighted"),
        "f1": f1_score(y_true, y_pred, average="weighted"),
        "auc": roc_auc_score(y_true, y_scores, multi_class="ovo", average="weighted"),
    }
    metrics.append(fold_metrics)
    
    if epoch_acc > best_val_accuracy:
        best_val_accuracy = epoch_acc
        best_model_metrics = fold_metrics
        best_model_state = model.state_dict()
        best_fold = fold
        best_y_true, best_y_pred = y_true, y_pred
    
    results.append({
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy
    })
    

In [ ]:
# Calculate average metrics
avg_metrics = {k: np.mean([m[k] for m in metrics]) for k in metrics[0].keys()}
print("Average Metrics across folds:", avg_metrics)
print("Best Fold:", best_fold, "Metrics:", best_model_metrics)

# Save best model
torch.save(best_model_state, "best_model.pth")

# Plot confusion matrix for the best model
conf_mat = confusion_matrix(best_y_true, best_y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=conf_mat)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix for Best Model")
plt.savefig("confusion_matrix.pdf")


# Plot average accuracy
avg_train_acc = np.mean([np.array(result["train_accuracy"]) for result in results], axis=0)
avg_val_acc = np.mean([np.array(result["val_accuracy"]) for result in results], axis=0)
plt.figure()
plt.plot(avg_train_acc, label="Train Accuracy")
plt.plot(avg_val_acc, label="Validation Accuracy")
plt.title("Average Accuracy Across Folds")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.savefig("average_accuracy.pdf")

# Plot best model accuracy
best_result = results[best_fold]
plt.figure()
plt.plot(best_result["train_accuracy"], label="Train Accuracy")
plt.plot(best_result["val_accuracy"], label="Validation Accuracy")
plt.title("Best Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.savefig("best_model_accuracy.pdf")

# Plot average loss
avg_train_loss = np.mean([np.array(result["train_loss"]) for result in results], axis=0)
avg_val_loss = np.mean([np.array(result["val_loss"]) for result in results], axis=0)
plt.figure()
plt.plot(avg_train_loss, label="Train Loss")
plt.plot(avg_val_loss, label="Validation Loss")
plt.title("Average Loss Across Folds")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.savefig("average_loss.pdf")

print("All plots saved as PDF.")

# Visualize features

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Function to extract features and labels using the best model
def extract_features(model, dataloader):
    model.eval()  # Set the model to evaluation mode
    features, labels = [], []
    with torch.no_grad():
        for inputs, label_batch in dataloader:
            inputs = inputs.cuda()
            label_batch = label_batch.cuda()
            # Forward pass to get features (use model's penultimate layer or embeddings)
            output = model(inputs)
            feature_batch = output.logits  # Change this if model has a specific feature extraction layer
            features.append(feature_batch.cpu().numpy())
            labels.append(label_batch.cpu().numpy())
    
    # Concatenate all features and labels
    features = np.concatenate(features, axis=0)
    labels = np.concatenate(labels, axis=0)
    return features, labels

In [ ]:
# Extract features and labels from validation set using the best model
best_model = lora_model.cuda()
best_model.load_state_dict(best_model_state)
features, labels = extract_features(best_model, val_loader)

# Perform t-SNE
tsne = TSNE(n_components=2, random_state=42,perplexity = 20)
tsne_features = tsne.fit_transform(features)

# Plot t-SNE features
plt.figure(figsize=(10, 8))
scatter = plt.scatter(tsne_features[:, 0], tsne_features[:, 1], c=labels, cmap="viridis", alpha=0.7)
plt.colorbar(scatter, label="Class Label")
# plt.title("t-SNE Visualization of Features (2 Components)")
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")

# Save plot as PDF
plt.savefig("tsne_visualization.pdf")
plt.show()